# Machine Learning Lab — Activities 1–6

**Activities are kept separate. Each task has its own code cell, and explanations/comments are placed in Markdown.**

> Solutions are written as alternative implementations rather than copying the activity wording/code structure directly from the lab.

## Activity 1 — Pakistani Provinces Analysis

**Tasks covered:** dataset creation, missing-value handling, label + one-hot encoding, annotated scatter plot, and IQR outlier detection.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

provinces = pd.DataFrame({
    "Province": ["Punjab", "Sindh", "Khyber Pakhtunkhwa", "Balochistan"],
    "Population": [127.69, 55.70, 40.86, 14.89],
    "Literacy_Rate": [64.7, 57.5, 55.1, 54.0],
    "Region": ["East", "South", "North", "West"]
})

print(provinces)

### Activity 1 — Missing Values

Create missing values so both required strategies can be demonstrated: dropping one incomplete row and filling another missing value with the median.

In [ ]:
# Artificially introduce two missing values for demonstration.
provinces.loc[1, "Literacy_Rate"] = np.nan
provinces.loc[2, "Population"] = np.nan

# Drop one row, then fill the remaining population value with the median.
provinces_clean = provinces.drop(index=3).reset_index(drop=True)
provinces_clean["Population"] = provinces_clean["Population"].fillna(
    provinces_clean["Population"].median()
)

print(provinces_clean)

### Activity 1 — Region Encoding

Use a manual numeric mapping for label encoding and `get_dummies()` for one-hot encoding.

In [ ]:
region_map = {"North": 0, "South": 1, "East": 2, "West": 3}
provinces_clean["Region_Label"] = provinces_clean["Region"].map(region_map)

region_one_hot = pd.get_dummies(
    provinces_clean["Region"], prefix="Region", dtype=int
)

provinces_encoded = pd.concat([provinces_clean, region_one_hot], axis=1)
print(provinces_encoded)

### Activity 1 — Population vs Literacy Rate

Plot population against literacy rate and label each visible province.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    provinces_encoded["Population"],
    provinces_encoded["Literacy_Rate"]
)

for _, row in provinces_encoded.iterrows():
    plt.annotate(
        row["Province"],
        (row["Population"], row["Literacy_Rate"]),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.xlabel("Population (millions)")
plt.ylabel("Literacy Rate (%)")
plt.title("Population vs Literacy Rate")
plt.grid(alpha=0.25)
plt.show()

### Activity 1 — Literacy-Rate Outliers

Use the IQR rule to check whether any province lies outside the normal range.

In [ ]:
q1 = provinces_encoded["Literacy_Rate"].quantile(0.25)
q3 = provinces_encoded["Literacy_Rate"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

literacy_outliers = provinces_encoded[
    (provinces_encoded["Literacy_Rate"] < lower) |
    (provinces_encoded["Literacy_Rate"] > upper)
]

print("IQR bounds:", lower, "to", upper)
print(literacy_outliers[["Province", "Literacy_Rate"]])

## Activity 2 — Student Performance Tracker

**Tasks covered:** generate 100 students, mean-impute missing marks, encode grades, histogram, boxplot, and grade-wise summary.

In [ ]:
rng = np.random.default_rng(42)

students = pd.DataFrame({
    "ID": np.arange(1, 101),
    "Math": rng.integers(35, 101, 100),
    "Science": rng.integers(40, 101, 100),
    "English": rng.integers(30, 101, 100)
})

students["Grade"] = pd.cut(
    students[["Math", "Science", "English"]].mean(axis=1),
    bins=[-np.inf, 59, 69, np.inf],
    labels=["C", "B", "A"]
)

print(students.head())

### Activity 2 — Missing Marks

Introduce a few missing marks and replace each missing subject value with that subject's mean.

In [ ]:
students.loc[[7, 31, 72], "Math"] = np.nan

for subject in ["Math", "Science", "English"]:
    students[subject] = students[subject].fillna(students[subject].mean())

print("Remaining missing values:")
print(students[["Math", "Science", "English"]].isna().sum())

### Activity 2 — Grade Encoding and Total Score

Map the ordered grades to numeric values and calculate the total of the three subjects.

In [ ]:
grade_map = {"C": 0, "B": 1, "A": 2}
students["Grade_Encoded"] = students["Grade"].map(grade_map)
students["Total"] = students[["Math", "Science", "English"]].sum(axis=1)

print(students.head(10))

### Activity 2 — Math Histogram

Use exactly 10 bins as required by the activity.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(students["Math"], bins=10, edgecolor="black")
plt.xlabel("Math Score")
plt.ylabel("Number of Students")
plt.title("Distribution of Math Scores")
plt.show()

### Activity 2 — Total Score Boxplot and Summary

The boxplot helps identify unusually high or low total scores. The group means provide a simple grade-wise performance summary.

In [ ]:
plt.figure(figsize=(7, 4))
plt.boxplot(students["Total"], vert=False)
plt.xlabel("Total Score")
plt.title("Total Score Outlier Check")
plt.show()

q1 = students["Total"].quantile(0.25)
q3 = students["Total"].quantile(0.75)
iqr = q3 - q1

outliers = students[
    (students["Total"] < q1 - 1.5 * iqr) |
    (students["Total"] > q3 + 1.5 * iqr)
]

print("Unusually high/low totals:")
print(outliers[["ID", "Total", "Grade"]])

print("\nAverage total by grade:")
print(students.groupby("Grade", observed=True)["Total"].mean())

## Activity 3 — COVID-19 Dataset Exploration

The dataset URL and Pakistan filter follow the supplied lab activity. This section uses daily changes in confirmed cases for the time-series and outlier analysis.

In [ ]:
covid_url = (
    "https://raw.githubusercontent.com/datasets/covid-19/master/"
    "data/time-series-19-covid-combined.csv"
)

covid = pd.read_csv(covid_url)
covid["Date"] = pd.to_datetime(covid["Date"])

pakistan = covid[covid["Country/Region"].eq("Pakistan")].copy()
pakistan = pakistan.sort_values("Date")

print(pakistan.head())

### Activity 3 — Missing Values and Daily Data

If more than one geographic record exists for a date, aggregate the records first. Then calculate daily confirmed cases from cumulative confirmed cases.

In [ ]:
daily = pakistan.groupby("Date", as_index=False).agg(
    Confirmed=("Confirmed", "sum"),
    Recovered=("Recovered", "sum"),
    Deaths=("Deaths", "sum")
)

daily["Daily_Cases"] = daily["Confirmed"].diff().fillna(daily["Confirmed"])
daily["Daily_Cases"] = daily["Daily_Cases"].clip(lower=0)

# Forward-fill any gaps, then use 0 for any values still missing.
daily = daily.ffill().fillna(0)

print("Missing values after cleaning:")
print(daily.isna().sum())

### Activity 3 — Cases Over Time

Plot the daily confirmed cases for Pakistan.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(daily["Date"], daily["Daily_Cases"])
plt.xlabel("Date")
plt.ylabel("Daily Confirmed Cases")
plt.title("Pakistan — Daily Confirmed COVID-19 Cases")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Activity 3 — Highest-Case Day and Outliers

In [ ]:
max_day = daily.loc[daily["Daily_Cases"].idxmax()]
print("Day with highest reported daily confirmed cases:")
print(max_day[["Date", "Daily_Cases"]])

plt.figure(figsize=(7, 4))
plt.boxplot(daily["Daily_Cases"], vert=False)
plt.xlabel("Daily Confirmed Cases")
plt.title("Outlier Detection in Daily Cases")
plt.show()

q1 = daily["Daily_Cases"].quantile(0.25)
q3 = daily["Daily_Cases"].quantile(0.75)
iqr = q3 - q1
outliers = daily[
    (daily["Daily_Cases"] < q1 - 1.5 * iqr) |
    (daily["Daily_Cases"] > q3 + 1.5 * iqr)
]
print("Number of IQR outlier days:", len(outliers))

## Activity 4 — Multi-format Data Loading & Visualization Challenge

Place `students.csv`, `attendance.json`, and `extra.xlsx` in the same folder as the notebook before running these cells.

In [ ]:
students_file = pd.read_csv("students.csv")
attendance_file = pd.read_json("attendance.json")
extra_file = pd.read_excel("extra.xlsx")

print("Students:")
print(students_file.head())
print("\nAttendance:")
print(attendance_file.head())
print("\nBonus:")
print(extra_file.head())

### Activity 4 — Standardize Columns and Merge

Clean whitespace/capitalization in column names, then perform two merges using `Name` as the key.

In [ ]:
for frame in [students_file, attendance_file, extra_file]:
    frame.columns = [c.strip().title() for c in frame.columns]

combined = (
    students_file
    .merge(attendance_file, on="Name", how="inner")
    .merge(extra_file, on="Name", how="inner")
)

combined = combined[["Name", "Marks", "Attendance", "Bonus"]]
print(combined)

### Activity 4 — Marks vs Attendance

Students with attendance below 70% are displayed separately so the threshold is visually clear.

In [ ]:
low_attendance = combined["Attendance"] < 70

plt.figure(figsize=(8, 5))
plt.scatter(
    combined.loc[~low_attendance, "Marks"],
    combined.loc[~low_attendance, "Attendance"],
    label="Attendance >= 70%"
)
plt.scatter(
    combined.loc[low_attendance, "Marks"],
    combined.loc[low_attendance, "Attendance"],
    label="Attendance < 70%"
)
plt.xlabel("Marks")
plt.ylabel("Attendance (%)")
plt.title("Marks vs Attendance")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

### Activity 4 — One-Hot Encoding of Name

In [ ]:
name_one_hot = pd.get_dummies(
    combined["Name"], prefix="Student", dtype=int
)
combined_encoded = pd.concat(
    [combined.drop(columns="Name"), name_one_hot], axis=1
)

print(combined_encoded)

## Activity 5 — Data Cleaning & Outlier Detection with Visualization

Use the exact dataset supplied in the activity.

In [ ]:
employee_data = pd.DataFrame({
    "Name": ["Ahsan", "Hira", "Bilal", "Zara", "Salman", "Mahnoor"],
    "Age": [25, 27, 35, 29, None, 40],
    "Salary": [50000, None, 75000, 2000000, 60000, 90000],
    "Department": ["IT", "Finance", "IT", "HR", "Finance", "IT"]
})

print(employee_data)

### Activity 5 — Missing Values

Use median imputation. For this small dataset, median is less sensitive to the extreme salary value than the mean.

In [ ]:
employee_data["Age"] = employee_data["Age"].fillna(employee_data["Age"].median())
employee_data["Salary"] = employee_data["Salary"].fillna(employee_data["Salary"].median())

print(employee_data)

### Activity 5 — Department Encoding

Apply one-hot encoding to the categorical Department column.

In [ ]:
employee_encoded = pd.get_dummies(
    employee_data, columns=["Department"], dtype=int
)
print(employee_encoded)

### Activity 5 — Salary Outlier Detection

Use a boxplot together with the IQR rule to identify statistically unusual salaries.

In [ ]:
q1 = employee_data["Salary"].quantile(0.25)
q3 = employee_data["Salary"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

salary_outliers = employee_data[
    (employee_data["Salary"] < lower) |
    (employee_data["Salary"] > upper)
]

plt.figure(figsize=(8, 4))
plt.boxplot(employee_data["Salary"], vert=False)
plt.xlabel("Salary")
plt.title("Salary Outlier Detection")
plt.show()

print("IQR bounds:", lower, "to", upper)
print("Outliers:")
print(salary_outliers[["Name", "Salary"]])

### Activity 5 — Zara's Salary

The IQR method can flag Zara statistically. Whether the value should actually be removed is a data-quality decision and requires checking whether the salary is a valid observation.

In [ ]:
zara_salary = employee_data.loc[
    employee_data["Name"].eq("Zara"), "Salary"
].iloc[0]

print("Zara's salary:", zara_salary)
print("Zara is an IQR outlier:", zara_salary > upper)

## Activity 6 — Breast Cancer Dataset Exploration

This activity follows the Iris-style dataset exploration requested in the lab, then performs missing-value handling, class-balance checking, outlier detection, transformation, scaling, PCA, and two-classifier comparison.

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()

print("data.keys():", cancer.keys())
print("target_names:", cancer.target_names)

n_samples, n_features = cancer.data.shape
print("Number of samples:", n_samples)
print("Number of features:", n_features)
print("feature_names:", cancer.feature_names)
print("data.data shape:", cancer.data.shape)
print("data.target shape:", cancer.target.shape)

### Activity 6 — Create DataFrame and Introduce Missing Values

The dataset has no missing values by default. Five random cells in `mean radius` are therefore set to `NaN`, as required by the activity.

In [ ]:
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df["target"] = cancer.target

print("Missing values before insertion:", df.isna().sum().sum())

rng = np.random.default_rng(7)
missing_rows = rng.choice(df.index, size=5, replace=False)
df.loc[missing_rows, "mean radius"] = np.nan

print("Rows changed:", missing_rows.tolist())
print("Missing mean-radius values:", df["mean radius"].isna().sum())

### Activity 6 — Handle Missing Mean Radius

Use the median because it is more robust to unusually large or small observations than the mean.

In [ ]:
radius_median = df["mean radius"].median()
df["mean radius"] = df["mean radius"].fillna(radius_median)

print("Median used:", radius_median)
print("Remaining missing mean-radius values:", df["mean radius"].isna().sum())

### Activity 6 — Class Distribution and Imbalance

Check the number of malignant and benign samples. Instead of generating synthetic observations, this solution uses class weighting during SVM training as an imbalance-handling alternative.

In [ ]:
class_counts = df["target"].value_counts().sort_index()
print("Class counts:")
print(class_counts)
print("Class labels:", dict(enumerate(cancer.target_names)))

imbalance_ratio = class_counts.max() / class_counts.min()
print("Largest/smallest class ratio:", round(imbalance_ratio, 3))

if imbalance_ratio > 1.5:
    print("The classes show a noticeable imbalance.")
else:
    print("The classes are relatively close in size.")

### Activity 6 — Prepare Data, Split, and Scale

Standard scaling puts features on comparable scales, which is particularly important for distance-based methods such as KNN and for SVM.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

### Activity 6 — IQR Outliers in Two Features

Check `mean radius` and `mean texture` separately with the IQR rule and visualize both distributions with a boxplot.

In [ ]:
feature_a = "mean radius"
feature_b = "mean texture"

plt.figure(figsize=(8, 4))
plt.boxplot(
    [df[feature_a], df[feature_b]],
    tick_labels=[feature_a, feature_b]
)
plt.ylabel("Feature value")
plt.title("IQR Outlier Check")
plt.show()

for feature in [feature_a, feature_b]:
    q1 = df[feature].quantile(0.25)
    q3 = df[feature].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    count = ((df[feature] < low) | (df[feature] > high)).sum()
    print(f"{feature}: {count} IQR outliers")

### Activity 6 — Log Transform Mean Area

Compare the distribution of `mean area` before and after `log1p` transformation.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["mean area"], bins=25, edgecolor="black")
plt.xlabel("Mean Area")
plt.ylabel("Frequency")
plt.title("Mean Area Before Log Transformation")
plt.show()

df["log_mean_area"] = np.log1p(df["mean area"])

plt.figure(figsize=(8, 4))
plt.hist(df["log_mean_area"], bins=25, edgecolor="black")
plt.xlabel("log(1 + Mean Area)")
plt.ylabel("Frequency")
plt.title("Mean Area After Log Transformation")
plt.show()

### Activity 6 — PCA to Two Components

PCA reduces the scaled feature set to two dimensions for visualization. The two components can be inspected to see how much variance they retain and how the target classes appear in the projection.

In [ ]:
from sklearn.decomposition import PCA

# Fit PCA on the complete scaled feature matrix for visualization.
X_scaled_all = StandardScaler().fit_transform(X)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled_all)

plt.figure(figsize=(8, 5))
for class_id, class_name in enumerate(cancer.target_names):
    mask = y.to_numpy() == class_id
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=class_name)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Breast Cancer Data — PCA Projection")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance in 2 components:", pca.explained_variance_ratio_.sum())

### Activity 6 — Train and Compare Two Classifiers

Use KNN and SVM. SVM receives `class_weight="balanced"` as the imbalance-handling step; KNN does not have a direct class-weight parameter.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
knn_pred = knn.predict(X_test_scaled)

svm = SVC(kernel="rbf", class_weight="balanced", random_state=42)
svm.fit(X_train_scaled, y_train)
svm_pred = svm.predict(X_test_scaled)

print("KNN accuracy:", accuracy_score(y_test, knn_pred))
print("SVM accuracy:", accuracy_score(y_test, svm_pred))

print("\nKNN confusion matrix:")
print(confusion_matrix(y_test, knn_pred))
print("\nSVM confusion matrix:")
print(confusion_matrix(y_test, svm_pred))

print("\nKNN classification report:")
print(classification_report(y_test, knn_pred, target_names=cancer.target_names))
print("\nSVM classification report:")
print(classification_report(y_test, svm_pred, target_names=cancer.target_names))

### Activity 6 — Preprocessing Impact

For this dataset, scaling is especially important for KNN and SVM because these methods depend on feature magnitudes/distances. The exact impact should be verified experimentally by comparing model results with and without the preprocessing step.

In [ ]:
print(
    "Scaling is a key preprocessing step here because KNN uses distances "
    "and SVM is sensitive to feature scale. Outlier handling, PCA, and "
    "imbalance correction can also affect results, so their impact should "
    "be measured by controlled comparisons rather than assumed."
)